# MPPP — 04 Error analysis of COLMAP alignments

Takes one or more alignments made by notebook 03 (their `colmap/error_input/` folders) and measures the numbers the error model (`mppp.error`) runs on, next to the values it assumes (`ModelConfig` defaults and the archive's `CASE_PARAMS`):

| section | measured | model value |
|---|---|---|
| 2 image precision | ε = rms residual / √2 per instrument, same-station vs cross-station tracks, against range | ε_intra 0.169 px (archive, pooled) |
| 3 cross-station gate | match rate vs convergence angle θ and \|ΔLMST\|: A (1 + θ/θ_c)^−k · exp(−ΔL/L0), θ_c = θ̄/CV², k = 1/CV² | A 0.4, θ̄ 4.3°, CV 0.36, L0 2.3 h |
| 4 decorrelation | ρ(θ) of pairwise triangulation residuals | ρ_0 0.22, θ_c 0.4°, exponential |
| 5 view graph | shared tracks between stations, connectivity | — |
| 6 registration | bundle-adjusted pose − telemetry prior, per station | 0.3 m (telemetry registration) |

**How the gate is measured.** For a random sample of reconstructed points, every image pair in which the point is geometrically visible (in front of both cameras, inside both frames) is a candidate. A trial is: the point is matched in image *a*; is it also matched in image *b*? The same-station rate is the reference and A is relative to it, as in the model. Terrain that never matched in either image (occluded, masked rover, no texture) stays out of the denominator (`pair_survival(conditional=False)` counts it too).

**Caveats.** Sparse tracks are biased towards well-matched terrain, so ε is a lower bound (post-fit residuals) and the survival rates are upper bounds. The gate is only measured where the rate actually falls inside 0–`THETA_MAX_DEG`; the fit reports `constrained = False` otherwise. A needs pairs at small \|ΔLMST\| and L0 needs a spread of \|ΔLMST\|: with few stations, hold L0 (`FIT_L0 = False`).

Figures and tables go to `OUT`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import sys, json, time, datetime
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import numpy as np
import matplotlib.pyplot as plt
import mppp
from mppp.error import alignment as EA
from mppp.error.core import correlation_kernel
from mppp.error.viewgraph import graph_report
print("mppp", mppp.__version__, "from", Path(mppp.__file__).parent)

try:
    import pandas as pd
    from IPython.display import display
    def show(rows, cols=None, digits=3):
        df = pd.DataFrame(rows)
        if cols:
            df = df[[c for c in cols if c in df]]
        display(df.round(digits))
        return df
except ImportError:                                   # plain printout without pandas
    pd = None
    def show(rows, cols=None, digits=3):
        cols = cols or list(rows[0]) if rows else []
        print("  ".join(f"{c:>14}" for c in cols))
        for r in rows:
            print("  ".join(f"{r.get(c):>14.{digits}f}" if isinstance(r.get(c), float) else f"{str(r.get(c)):>14}"
                            for c in cols))
        return rows

## 1  Alignments

`ALIGNMENTS`: a name and a folder per alignment — the WORK folder of notebook 03, its `colmap` folder or the `error_input` folder itself. Missing ones are skipped. Exports from before 0.14.5 work too (they have no Mastcam-Z instrument split).

`MIN_TRACK_LENGTH` (default 3) keeps only tie points seen in at least that many images. The two-view points (about half of all points, mostly single stereo pairs) are then left out. That applies to every section below: ε, the gate, decorrelation, the view graph. Restricting to longer tracks also raises the conditional match rates, because a point seen in three images is more likely to be seen in a fourth; compare with `MIN_TRACK_LENGTH = 2` before reading the gate as absolute.

In [ ]:
ALIGNMENTS = {
    "Three Forks Navcam":             r"D:\scapes\threeforks_colmap",
    "Three Forks Navcam + Mastcam-Z": r"D:\scapes\threeforks_colmap_nav_zcam34",
    "Belva Navcam":                   r"D:\scapes\belva_colmap",
    "Rockytop Navcam":                r"D:\scapes\rockytop_colmap",
    "Bell Island Navcam":             r"D:\scapes\bellisland_colmap",
    "Taylor Fjellet Navcam":          r"D:\scapes\taylorfjellet_colmap",
}
MIN_TRACK_LENGTH = 3     # v0p22: only tie points seen in >= 3 images (2 = all, incl. the two-view points)
N_POINTS      = 5000     # points sampled per alignment for the gate (about 10 s per 1000 points for 300 images)
N_BOOT        = 50       # bootstrap resamples of those points for the gate's 16-84 % range (0 = none)
THETA_MAX_DEG = 30.0     # the gate is fitted over 0 .. this convergence angle
FIT_L0        = False    # also fit the illumination e-fold L0 (needs cross-station pairs over a range of |dLMST|)
SEED          = 0
OUT = Path(r"D:\scapes\error_analysis") / datetime.date.today().isoformat()

In [ ]:
OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)
als = {}
for name, path in ALIGNMENTS.items():
    try:
        t = time.time()
        als[name] = EA.load_alignment(path, label=name, min_track_length=MIN_TRACK_LENGTH)
        print(f"{name}: {len(als[name].model.images)} images, {len(als[name].model.points)} points "
              f"(of {als[name].points_before_track_filter}; tracks >= {MIN_TRACK_LENGTH} images), "
              f"{len(als[name].stations)} stations  ({time.time() - t:.0f} s)  <- {als[name].root}")
    except FileNotFoundError as e:
        print(f"{name}: skipped ({e})")
assert als, "no alignment found"
print("output:", OUT)

In [ ]:
rows = []
for name, al in als.items():
    s, tr = al.summary, al.summary.get("tracks", {})
    fams = sorted({r["family"] for r in al.images.values()})
    rows.append({"alignment": name, "cameras": "+".join(fams), "images": len(al.model.images),
                 "registered": s.get("registered_images"), "stations": len(al.stations), "points": len(al.model.points),
                 "cross_station_fraction": tr.get("cross_station_fraction"), "mean_track_length": tr.get("mean_track_length"),
                 "residual_rms_px": s.get("residual_rms_native_px"), "components": len(s.get("station_components") or [])})
overview = show(rows)

## 2  Image-measurement precision ε

ε = rms(reprojection residual)/√2: one image axis, in native pixels (the model's `per_image` convention). The residuals are post-fit. `eps_dof_px` corrects for the three coordinates each point absorbs: ε·√(2N/(2N − 3P)) for N observations of P points. That makes alignments with different track lengths comparable. "intra" = tracks inside one station, "cross" = tracks that span stations. The archive measured ε flat in θ, so ε_cross ≈ ε_intra is expected; the model ties ε_cross to ε_intra.

In [ ]:
eps_rows = [r for al in als.values() for r in EA.eps_table(al)]
eps = show(eps_rows, ["alignment", "group", "tracks", "n_obs", "n_points", "eps_px", "eps_dof_px", "rms_px", "median_px"])

fig, axes = plt.subplots(1, len(als), figsize=(4.2 * len(als), 3.4), squeeze=False, sharey=True)
for ax, (name, al) in zip(axes[0], als.items()):
    fams = sorted({r["family"] for r in al.images.values()})
    for fam in fams:
        er = EA.eps_by_range(al, family=fam)
        for kind, ls in (("intra", "-"), ("cross", "--")):
            ok = er[f"n_{kind}"] >= 100
            ax.plot(er["range_m"][ok], er[f"eps_{kind}_px"][ok], ls, marker="o", ms=3, label=f"{fam} {kind}")
    ax.axhline(0.169, color="0.5", lw=0.8, ls=":", label="archive 0.169 px")
    ax.set_title(name, fontsize=10); ax.set_xlabel("range [m]"); ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("ε [native px]"); axes[0, 0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "eps_vs_range.png", dpi=150); plt.show()

## 3  Cross-station gate

Per alignment: the same-station rate (reference, grey) and the cross-station rate against convergence angle, with the fitted gate and the shapes of the archive cases (their θ̄ and CV with the measured A; all evaluated at the pairs' own |ΔLMST|). With Mastcam-Z in the alignment, the gate is also fitted per camera pairing.

Read `theta_half_deg` first: the angle at which the cross-station rate has halved, whatever the shape. `constrained = False` means the power law ran to a limit of its form (CV → 0: a sharper cut-off than the model allows); `chi2_dof` says how well the form fits (the archive rejected the Gaussian gate at χ²/dof 38–855). The archive's A (0.4) was defined against its own denominator, so compare θ̄, CV and θ_half, and treat A as this notebook's own normalisation.

In [ ]:
pairs = {}
for name, al in als.items():
    t = time.time()
    pairs[name] = EA.pair_survival(al, n_points=N_POINTS, seed=SEED)
    p = pairs[name]
    print(f"{name}: {p['observed'].size} candidate pairs ({int(p['cross'].sum())} cross-station) "
          f"from {p['n_points']} points, {time.time() - t:.0f} s")

In [ ]:
MIN_TRIALS = 2000          # cross-station trials needed to fit a camera pairing
fits = []
for name, p in pairs.items():
    fams = [None] + sorted({f for f in np.unique(p["families"]) if f and len(np.unique(p["families"])) > 1})
    for fam in fams:
        sel = EA._select(p, fam, True)
        if p["weight"][sel].sum() < MIN_TRIALS:
            continue
        g = EA.fit_gate(p, families=fam, fit_L0=FIT_L0, theta_max_deg=THETA_MAX_DEG, n_boot=N_BOOT, seed=SEED)
        fits.append(dict(g, alignment=name))
# pooled over the alignments, Navcam-Navcam pairs (the archive's stratum)
if len(pairs) > 1:
    pooled = EA.combine_pairs(list(pairs.values()))
    g = EA.fit_gate(pooled, families="Navcam-Navcam", fit_L0=FIT_L0, theta_max_deg=THETA_MAX_DEG, n_boot=N_BOOT, seed=SEED)
    fits.append(dict(g, alignment="pooled"))
gate_cols = ["alignment", "families", "theta_half_deg", "constrained", "chi2_dof", "A", "A_16_84", "theta_bar_deg",
             "theta_bar_deg_16_84", "cv", "cv_16_84", "L0_h", "s_intra", "A_absolute", "n_cross_pairs", "matched_cross_pairs",
             "dlmst_h_range"]
gates = show(fits, gate_cols)
defaults = EA.model_defaults()
print("model / archive 'measured': A %.2f, theta_bar %.1f deg, CV %.2f, L0 %.1f h" %
      (defaults["gate_A"], defaults["theta_bar_deg"], defaults["gate_cv"], defaults["L0_h"]))
bad = [f"{f['alignment']} ({f['families']})" for f in fits if "A" in f and not f["constrained"]]
if bad:
    print("power law at a limit of its form (CV -> 0 or theta_bar >= %g deg): %s -- the decay is not the model's shape; "
          "compare theta_half_deg instead" % (THETA_MAX_DEG, ", ".join(bad)))
poor = [f"{f['alignment']} ({f['families']}): {f['chi2_dof']:.0f}" for f in fits if "A" in f and f["chi2_dof"] > 10]
if poor:
    print("chi2/dof > 10 (form fits poorly; the archive rejected the Gaussian gate at 38-855):", "; ".join(poor))

In [ ]:
edges = np.arange(0, THETA_MAX_DEG + 1, 1.0)
case_style = {"pessimistic": ":", "measured": "--", "optimistic": "-."}
fig, axes = plt.subplots(1, len(pairs), figsize=(4.4 * len(pairs), 3.6), squeeze=False)
for ax, (name, p) in zip(axes[0], pairs.items()):
    intra = EA.survival_curve(p, edges, cross=False)
    cross = EA.survival_curve(p, edges, cross=True)
    ok = intra["n_possible"] >= 50
    ax.plot(intra["theta_deg"][ok], intra["rate"][ok], "o", color="0.6", ms=3, label="same station")
    ok = cross["n_possible"] >= 50
    ax.errorbar(cross["theta_deg"][ok], cross["rate"][ok], [cross["rate"][ok] - cross["lo"][ok], cross["hi"][ok] - cross["rate"][ok]],
                fmt="o", ms=3, color="C0", label="cross station")
    f = next((x for x in fits if x["alignment"] == name and x["families"] == "all" and "A" in x), None)
    if f:
        pred = EA.gate_expected(p, f["s_intra"], f["A"], f["theta_bar_deg"], f["cv"], f["L0_h"], edges)
        ax.plot(cross["theta_deg"], pred, "-", color="C3",
                label=f"fit A {f['A']:.2f} θ̄ {f['theta_bar_deg']:.1f}° CV {f['cv']:.2f}" + ("" if f["constrained"] else " (unconstrained)"))
        for case, c in defaults["cases"].items():      # archive SHAPE (theta_bar, CV) with the measured A
            pc = EA.gate_expected(p, f["s_intra"], f["A"], c["theta_bar_deg"], c["cv"], f["L0_h"], edges)
            ax.plot(cross["theta_deg"], pc, case_style[case], color="0.3", lw=0.9,
                    label=f"archive {case} θ̄ {c['theta_bar_deg']}° CV {c['cv']}")
        if np.isfinite(f["theta_half_deg"]):
            ax.axvline(f["theta_half_deg"], color="C3", lw=0.6, ls=":")
    ax.set_title(name, fontsize=10); ax.set_xlabel("convergence angle θ [deg]"); ax.set_ylim(0, None); ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("P(matched in b | matched in a)"); axes[0, 0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "gate.png", dpi=150); plt.show()

In [ ]:
# |dLMST|: cross-station rate at small angle (theta < 5 deg) against the LMST difference of the two images
fig, ax = plt.subplots(figsize=(5.2, 3.6))
dl_edges = np.arange(0, 9, 0.5)
for name, p in pairs.items():
    sel = EA._select(p, None, True) & (p["theta_deg"] < 5)
    dl = np.nan_to_num(p["dlmst_h"][sel], nan=-1)
    k = np.digitize(dl, dl_edges) - 1
    ok = (k >= 0) & (k < dl_edges.size - 1)
    w = p["weight"][sel][ok].astype(float)
    n = np.bincount(k[ok], weights=w, minlength=dl_edges.size - 1)
    s = np.bincount(k[ok], weights=w * p["observed"][sel][ok], minlength=dl_edges.size - 1)
    m = n >= 100
    ax.plot(0.5 * (dl_edges[:-1] + dl_edges[1:])[m], (s / np.maximum(n, 1))[m], "o-", ms=3, label=name)
x = np.linspace(0, 8, 50)
ax.plot(x, 0.4 * np.exp(-x / defaults["L0_h"]), "k--", lw=0.9, label=f"exp(-ΔL/{defaults['L0_h']} h) (shape)")
ax.set_xlabel("|ΔLMST| [h]"); ax.set_ylabel("cross-station rate, θ < 5°"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "gate_dlmst.png", dpi=150); plt.show()

## 4  Decorrelation ρ(θ)

For tracks of ≥ 3 images: the point is triangulated from every image pair; ρ is the correlation of those pair residuals against the angle between the pairs' bisectors (`colmap.measure_theta_c`). The model uses an exponential kernel with ρ_0 = 0.22 at zero separation, decaying with θ_c = 0.4°, and ρ_∞ = 0.

In [ ]:
rhos = {}
fig, ax = plt.subplots(figsize=(5.2, 3.6))
for name, al in als.items():
    t = time.time()
    rhos[name] = r = EA.decorrelation(al, n_points=min(N_POINTS, 4000), seed=SEED)
    ok = r["n"] >= 50
    ax.plot(r["theta_deg"][ok], r["rho"][ok], "o-", ms=3, label=f"{name} ({r['n_tracks']} tracks)")
    print(f"{name}: {time.time() - t:.0f} s, median pair-triangulation residual {r['residual_scale_m'] * 1000:.3f} mm")
th = np.linspace(0, 5, 200)
ax.plot(th, correlation_kernel(th, defaults["theta_c_deg"], defaults["rho_inf"], defaults["correlation_kernel"], defaults["rho_0"]),
        "k--", lw=0.9, label="model (exponential, ρ_0 0.22, θ_c 0.4°)")
ax.set_xlabel("separation of pair bisectors [deg]"); ax.set_ylabel("ρ"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "rho.png", dpi=150); plt.show()

## 5  Measured view graph

Edge weight = tracks shared by two stations. λ₂ = 0 means the stations are not all tied together; articulation stations are single points of failure.

In [ ]:
graphs = {}
fig, axes = plt.subplots(1, len(als), figsize=(4.4 * len(als), 4.0), squeeze=False)
for ax, (name, al) in zip(axes[0], als.items()):
    graphs[name] = g = EA.view_graph(al)
    print(f"--- {name}\n" + graph_report(g))
    labels = [al.station_label(s) for s in g.names]
    im = ax.imshow(np.log10(1 + g.W), cmap="viridis")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=6)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=6)
    ax.set_title(name, fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.7, label="log10(1 + shared tracks)")
fig.tight_layout(); fig.savefig(OUT / "view_graph.png", dpi=150); plt.show()

## 6  Registration: bundle adjustment vs telemetry

Median over each station's images of |C_BA − C_prior| (images with < 30 observations left out). The BA is tied to the priors (σ 1 m by default), so these are lower bounds on the telemetry error. The model's FIXED case uses a registration error of 0.3 m.

In [ ]:
reg_rows = [r for al in als.values() for r in EA.registration(al)]
reg = show(reg_rows, ["alignment", "station_label", "images", "dC_median_m", "dE_median_m", "dN_median_m", "dU_median_m",
                      "dAttitude_median_deg"])
for name, al in als.items():
    for c in al.summary.get("prior_similarity_by_component", []):
        print(f"{name}: similarity prior -> BA over {len(c['stations'])} stations: scale {c['scale_refined_over_prior']:.4f}, "
              f"rotation {c['rotation_deg']:.2f} deg, residual per station {c['station_residual_after_similarity_m']}")

## 7  Summary

One row per alignment, beside the model's values. Where a gate is `constrained`, `model_config` gives the `ModelConfig` arguments to run the error model with the measured numbers.

In [ ]:
summary_rows = []
for name, al in als.items():
    g = next((f for f in fits if f["alignment"] == name and f["families"] in ("all", "Navcam-Navcam") and "A" in f), None)
    summary_rows.append(EA.parameters(al, g, rhos.get(name)))
summary = show(summary_rows)
d = {k: v for k, v in defaults.items() if k != "cases"}
print("model:", json.dumps(d))

model_config = {}
for f in fits:
    if "A" in f and f["constrained"]:
        model_config[f"{f['alignment']} [{f['families']}]"] = {"gate_A": round(f["A"], 3), "theta_bar_deg": round(f["theta_bar_deg"], 2),
                                                               "gate_cv": round(f["cv"], 3), "L0_h": round(f["L0_h"], 2)}
print(json.dumps(model_config, indent=1))

def _j(o):
    return o.item() if isinstance(o, np.generic) else o.tolist() if isinstance(o, np.ndarray) else str(o)
(OUT / "error_analysis.json").write_text(json.dumps({
    "mppp_version": mppp.__version__, "alignments": {k: str(v.root) for k, v in als.items()},
    "settings": {"MIN_TRACK_LENGTH": MIN_TRACK_LENGTH, "N_POINTS": N_POINTS, "N_BOOT": N_BOOT, "THETA_MAX_DEG": THETA_MAX_DEG, "FIT_L0": FIT_L0, "SEED": SEED},
    "eps": eps_rows, "gate_fits": fits, "registration": reg_rows, "summary": summary_rows, "model": defaults,
    "model_config": model_config,
    "rho": {k: {"theta_deg": v["theta_deg"], "rho": v["rho"], "n": v["n"]} for k, v in rhos.items()}}, indent=1, default=_j),
    encoding="utf-8")
if pd is not None:
    pd.DataFrame(summary_rows).to_csv(OUT / "summary.csv", index=False)
    pd.DataFrame(fits).to_csv(OUT / "gate_fits.csv", index=False)
    pd.DataFrame(eps_rows).to_csv(OUT / "eps.csv", index=False)
print("written:", sorted(p.name for p in OUT.iterdir()))